### Block 0 — Install packages

In [5]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn tqdm pandas huggingface_hub
!pip install -U bitsandbytes

In [2]:
from huggingface_hub import login

login()

### Block 1 — Imports and Gemini API client

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

from datasets import load_dataset
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import os

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print(f"Loading {MODEL_ID} in 4-bit quantization...")


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token_id = tokenizer.eos_token_id


model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=10,
    return_full_text=False,
    temperature=0.01,
    do_sample=False
)

print("Model loaded successfully!")

Loading meta-llama/Llama-3.1-8B-Instruct in 4-bit quantization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded successfully!


### Block 2 — Load FPB dataset

In [4]:
dataset = load_dataset("ChanceFocus/en-fpb", split="test")

print(dataset)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])


README.md:   0%|          | 0.00/742 [00:00<?, ?B/s]

data/train-00000-of-00001-ab9a3b4799b095(…):   0%|          | 0.00/608k [00:00<?, ?B/s]

data/test-00000-of-00001-8bd1e21c671fb67(…):   0%|          | 0.00/188k [00:00<?, ?B/s]

data/valid-00000-of-00001-303e4ba2afe838(…):   0%|          | 0.00/154k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/970 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/776 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 970
})
Number of examples: 970
Example 0: {'id': 'fpb3876', 'query': 'Analyze the sentiment of this statement extracted from a financial news article. Provide your answer as either negative, positive, or neutral.\nText: The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .\nAnswer:', 'answer': 'positive', 'text': 'The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from Larox to Etteplan .', 'choices': ['positive', 'neutral', 'negative'], 'gold': 0}


### Block 3 — Labels and normalization

In [5]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map the raw model output to one of: 'negative', 'neutral', 'positive'.
    """
    text = (raw or "").strip().lower()

    # Exact match first
    for label in LABELS:
        if label in text:
            return label

    # Handle short variants like "pos" or "neg"
    if "pos" in text:
        return "positive"
    if "neg" in text:
        return "negative"

    # Fallback
    return "neutral"


### Block 4 — Llama sentiment classifier

In [8]:
def predict_label(text: str) -> str:
    """
    Call Llama 3.1 (Local Pipeline) and return a normalized label.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert financial sentiment classifier. "
                "Classify the sentiment of the financial news snippet as exactly one of: negative, neutral, or positive. "
                "Return only one word."
            )
        },
        {
            "role": "user",
            "content": f"Text: {text}\nAnswer:"
        }
    ]


    outputs = text_generator(
        messages,
        pad_token_id=tokenizer.eos_token_id
    )


    raw = outputs[0]["generated_text"]


    return normalize_prediction(raw)

### Block 5 — Evaluation loop over FPB test split

In [9]:
y_true = []
y_pred = []

for example in tqdm(dataset, total=len(dataset)):
    text = example["text"]
    true_label = example["answer"].strip().lower()

    y_true.append(true_label)

    pred_label = predict_label(text)
    y_pred.append(pred_label)


100%|██████████| 970/970 [03:06<00:00,  5.19it/s]


### Block 6 — Compute metrics and inspect predictions

In [10]:
# Overall accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

# Build DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))


Accuracy: 0.7763

Classification report:
              precision    recall  f1-score   support

    negative     0.8250    0.8534    0.8390       116
     neutral     0.8428    0.7712    0.8054       577
    positive     0.6491    0.7545    0.6978       277

    accuracy                         0.7763       970
   macro avg     0.7723    0.7931    0.7807       970
weighted avg     0.7853    0.7763    0.7787       970


First 10 predictions:
                                                                                                                                                                                                                                                                                 text     gold     pred
                                                                                           The new agreement , which expands a long-established cooperation between the companies , involves the transfer of certain engineering and documentation functions from L